# Data Cleaning and Preprocessing

### 1. Import Libraries

In [1]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np

### 2. Exploration and Preprocessing

In [2]:
df_original = pd.read_csv('./datasets/full_dataset.csv')

df_original.head()

,Unnamed: 0,rk,player,nation,pos,squad,comp,age,born,Matches Played,...,Total Shots,% Shots on target,Shots p 90,Goals per shot,Goals per shot on target,% Aerial Duels won,Shot creating actions p 90,Goal creating actions p 90,Crosses Stopped,season
0,0,1,Patrick van Aanholt,Netherlands,DF,Crystal Palace,Premier League,26,1990,28,...,33,33.3,0.45,0.15,0.45,54.5,1.90,0.16,0.0,2017-2018
1,1,2,Rolando Aarons,England,MF,Newcastle Utd,Premier League,21,1995,4,...,2,0.0,0.00,0.00,0.00,25.0,0.65,0.00,0.0,2017-2018
2,2,3,Rolando Aarons,England,MF,Hellas Verona,Serie A,21,1995,11,...,3,0.0,0.00,0.00,0.00,37.5,0.87,0.00,0.0,2017-2018
3,3,4,Ignazio Abate,Italy,DF,Milan,Serie A,30,1986,17,...,4,50.0,0.17,0.25,0.50,55.6,2.30,0.34,0.0,2017-2018
4,4,5,Aymen Abdennour,Tunisia,DF,Marseille,Ligue 1,27,1989,8,...,2,50.0,0.18,0.00,0.00,50.0,0.36,0.00,0.0,2017-2018


In [3]:
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18243 entries, 0 to 18242
Data columns (total 66 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Unnamed: 0                    18243 non-null  int64  
 1   rk                            18243 non-null  int64  
 2   player                        18243 non-null  object 
 3   nation                        18243 non-null  object 
 4   pos                           18243 non-null  object 
 5   squad                         18243 non-null  object 
 6   comp                          18243 non-null  object 
 7   age                           18243 non-null  int64  
 8   born                          18243 non-null  int64  
 9   Matches Played                18243 non-null  int64  
 10  Avg Mins per Match            18243 non-null  int64  
 11  Goals                         18243 non-null  int64  
 12  Assists                       18243 non-null  int64  
 13  G

We can group our features in related groups
- Player Identity
- Club Info.
- Player Statistics (Playing Time - Attacking - Passing - Defending - Goalkeeping - Carrying & Dribblling - Shooting & Creation)

#### 2.1 Player Identity Features
Columns:
- player
- nation
- pos
- age
- born

In [4]:
df_original[['rk', 'player', 'pos', 'nation', 'age', 'born']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18243 entries, 0 to 18242
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   rk      18243 non-null  int64 
 1   player  18243 non-null  object
 2   pos     18243 non-null  object
 3   nation  18243 non-null  object
 4   age     18243 non-null  int64 
 5   born    18243 non-null  int64 
dtypes: int64(3), object(3)
memory usage: 855.3+ KB


In [5]:
# Remove "rk", since it's just the indexing of each season
# Remove "player", since it's not relevant info for the model
# Remove "born", since we already have the "age"
df_preprocessed = df_original.drop(columns=['rk', 'player', 'born'])

In [6]:
# Fix invalid/inconsistent nation column values
nation_fixes = {
    'gp GLP': 'Guadeloupe',
    'tt TRI': 'Trinidad and Tobago',
    'cy CYP': 'Cyprus',
    'gt GUA': 'Guatemala',
    'sr SUR': 'Suriname',
    'bo BOL': 'Bolivia',
    'cu CUB': 'Cuba',
    'tz TAN': 'Tanzania',
    'kn SKN': 'Saint Kitts and Nevis',
    'fo FRO': 'Faroe Islands',
    'pa PAN': 'Panama',
    'gd GRN': 'Grenada',
    'lt LTU': 'Lithuania',
    'mt MLT': 'Malta',
}
 
df_preprocessed['nation'] = df_preprocessed['nation'].replace(nation_fixes)

In [7]:
print(f'Number of distinct countries: {df_preprocessed['nation'].nunique()}')
df_preprocessed['nation'].value_counts()

Number of distinct countries: 132


nation
Spain           2654
France          2216
Italy           1644
Germany         1517
England         1234
                ... 
Bermuda            1
Saudi Arabia       1
North Korea        1
Laos               1
Malta              1
Name: count, Length: 132, dtype: int64

We have 2 categorical colunms (pos, nation). `pos` can be encoded using *One-Hot encoding*. `nation` has 132 unique values, so using *One-Hot encoding* with it will not be efficient. We can use *Target encoding* with it, but now I will use *Label encoding* as I did not select the target variable yet.

In [8]:
# "pos" : One-Hot Encoding

df_preprocessed = pd.get_dummies(df_preprocessed, columns=['pos'])
dummy_cols = [col for col in df_preprocessed.columns if col.startswith('pos')]
df_preprocessed[dummy_cols] = df_preprocessed[dummy_cols].astype('int')

In [9]:
# "nation" : Label Encoding

le = LabelEncoder()
df_preprocessed['nation'] = le.fit_transform(df_preprocessed['nation'])

#### 2.2 Club Info. Features
Columns:
- squad
- comp
- season

In [10]:
df_preprocessed[['squad', 'comp', 'season']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18243 entries, 0 to 18242
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   squad   18243 non-null  object
 1   comp    18243 non-null  object
 2   season  18243 non-null  object
dtypes: object(3)
memory usage: 427.7+ KB


In [11]:
# "season" : convert to integer number indicating the starting year

seasons = {'2017-2018': 2017,
           '2018-2019': 2018,
           '2019-2020': 2019,
           '2020-2021': 2020,
           '2021-2022': 2021,
           '2022-2023': 2022,
           '2023-2024': 2023
}

df_preprocessed['season'] = df_preprocessed['season'].replace(seasons)

C:\Users\CARNIVAL\AppData\Local\Temp\ipykernel_16280\1264396236.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_preprocessed['season'] = df_preprocessed['season'].replace(seasons)


In [12]:
# "squad" : Should be Tardet Encoding, but I will use Label Encoding per competition

le = LabelEncoder()
df_preprocessed['squad'] = df_preprocessed.groupby('comp')['squad'].transform(lambda x: le.fit_transform(x))

In [13]:
# "comp" : One-Hot Encoding

df_preprocessed = pd.get_dummies(df_preprocessed, columns=['comp'])
dummy_cols = [col for col in df_preprocessed.columns if col.startswith('comp')]
df_preprocessed[dummy_cols] = df_preprocessed[dummy_cols].astype('int')

#### 2.3 Player Statistics
We can divide these group into subgroups (Playing Time, Attacking, Passing, Defending, Goalkeeping, Carrying and Dribbling, Shooting and Creation)

##### 2.3.1 Playing Time
- Matches Played
- Avg Mins per Match

In [14]:
df_preprocessed[['Matches Played', 'Avg Mins per Match']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18243 entries, 0 to 18242
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Matches Played      18243 non-null  int64
 1   Avg Mins per Match  18243 non-null  int64
dtypes: int64(2)
memory usage: 285.2 KB


No preprocessing needed

##### 2.3.2 Attacking
- Goals
- Assists
- Goals & Assists
- Non Penalty Goals
- Penalty Kicks Made
- Expected Goals
- Exp NPG
- Goals p 90
- Assists p 90

In [15]:
df_preprocessed[['Goals', 'Assists', 'Goals & Assists', 'Non Penalty Goals', 'Penalty Kicks Made', 'Expected Goals', 'Exp NPG', 'Goals p 90', 'Assists p 90']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18243 entries, 0 to 18242
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Goals               18243 non-null  int64  
 1   Assists             18243 non-null  int64  
 2   Goals & Assists     18243 non-null  int64  
 3   Non Penalty Goals   18243 non-null  int64  
 4   Penalty Kicks Made  18243 non-null  int64  
 5   Expected Goals      18243 non-null  float64
 6   Exp NPG             18243 non-null  float64
 7   Goals p 90          18243 non-null  float64
 8   Assists p 90        18243 non-null  float64
dtypes: float64(4), int64(5)
memory usage: 1.3 MB


In [16]:
sum(df_preprocessed['Goals'] == df_preprocessed['Goals Scored'])

18243

In [17]:
# Remove "Goals and Assists" as we have "Goals" and "Assists"
# Remove "Goals Scored" as it's a duplicate column of "Goals"

df_preprocessed = df_preprocessed.drop(columns=['Goals & Assists', 'Goals Scored'])

##### 2.3.3 Passing
- Passes Completed
- Passes Attempted
- Pass completion %
- Progressive Passes
- Progressive passes distance
- % Short pass completed
- % Medium passes completed
- % Long passes completed
- Key passes
- 1/3
- Passes into penalty area

In [18]:
df_preprocessed[['Passes Completed', 'Passes Attempted', 'Pass completion %', 'Progressive Passes', 'Progressive passes distance', r'% Short pass completed', r'% Medium passes completed', r'% Long passes completed', 'Key passes', '1/3', 'Passes into penalty area']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18243 entries, 0 to 18242
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Passes Completed             18243 non-null  int64  
 1   Passes Attempted             18243 non-null  int64  
 2   Pass completion %            18243 non-null  float64
 3   Progressive Passes           18243 non-null  int64  
 4   Progressive passes distance  18243 non-null  int64  
 5   % Short pass completed       18243 non-null  float64
 6   % Medium passes completed    18243 non-null  float64
 7   % Long passes completed      18243 non-null  float64
 8   Key passes                   18243 non-null  int64  
 9   1/3                          18243 non-null  int64  
 10  Passes into penalty area     18243 non-null  int64  
dtypes: float64(4), int64(7)
memory usage: 1.5 MB


No preprocessing needed

##### 2.3.4 Defending
- Tackles attempted
- Tackles Won
- % Dribbles tackled
- Shots blocked
- Passes blocked
- Interceptions
- Clearances
- Errors made
- touches_def_pen

In [19]:
df_preprocessed[['Tackles attempted', 'Tackles Won', '% Dribbles tackled', 'Shots blocked', 'Passes blocked', 'Interceptions', 'Clearances', 'Errors made', 'touches_def_pen']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18243 entries, 0 to 18242
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Tackles attempted   18243 non-null  int64  
 1   Tackles Won         18243 non-null  int64  
 2   % Dribbles tackled  18243 non-null  float64
 3   Shots blocked       18243 non-null  int64  
 4   Passes blocked      18243 non-null  int64  
 5   Interceptions       18243 non-null  int64  
 6   Clearances          18243 non-null  int64  
 7   Errors made         18243 non-null  int64  
 8   touches_def_pen     18243 non-null  int64  
dtypes: float64(1), int64(8)
memory usage: 1.3 MB


No preprocessing needed

##### 2.3.5 Goalkeeping
- Goals Against
- Goals against p 90
- Saves
- Saves %
- Clean Sheets
- % Clean sheets
- % Penalty saves
- Crosses Stopped

In [20]:
df_preprocessed[['Goals Against', 'Goals against p 90', 'Saves', 'Saves %', 'Clean Sheets', '% Clean sheets', '% Penalty saves', 'Crosses Stopped']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18243 entries, 0 to 18242
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Goals Against       18243 non-null  int64  
 1   Goals against p 90  18243 non-null  float64
 2   Saves               18243 non-null  int64  
 3   Saves %             18243 non-null  float64
 4   Clean Sheets        18243 non-null  int64  
 5   % Clean sheets      18243 non-null  float64
 6   % Penalty saves     18243 non-null  float64
 7   Crosses Stopped     18243 non-null  float64
dtypes: float64(5), int64(3)
memory usage: 1.1 MB


No preprocessing needed

##### 2.3.6 Carrying and Dribbling
- Progressive Carries 
- carries final 3rd
- carries penalty area
- Take ons attempted
- % Successful take-ons
- Times tackled during take-on
- Possessions lost

In [21]:
df_preprocessed[['Progressive Carries', 'carries final 3rd', 'carries penalty area', 'Take ons attempted', '% Successful take-ons', 'Times tackled during take-on', 'Possessions lost']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18243 entries, 0 to 18242
Data columns (total 7 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Progressive Carries           18243 non-null  int64  
 1   carries final 3rd             18243 non-null  int64  
 2   carries penalty area          18243 non-null  int64  
 3   Take ons attempted            18243 non-null  int64  
 4   % Successful take-ons         18243 non-null  float64
 5   Times tackled during take-on  18243 non-null  float64
 6   Possessions lost              18243 non-null  int64  
dtypes: float64(2), int64(5)
memory usage: 997.8 KB


In [22]:
sum(df_preprocessed['Progressive Carries'] == df_preprocessed['carries_prgc'])

18243

In [23]:
# Remove "carries_prgc" as it's a duplicate column of "Progressive Carries"
df_preprocessed = df_preprocessed.drop(columns=['carries_prgc'])

##### 2.3.7 Shooting and Creation
- Total Shots
- % Shots on target
- Shots p 90
- Goals per shot
- Goals per shot on target
- % Aerial Duels won
- Shot creating actions p 90
- Goal creating actions p 90

In [24]:
df_preprocessed[['Total Shots', '% Shots on target', 'Shots p 90', 'Goals per shot', 'Goals per shot on target', '% Aerial Duels won', 'Shot creating actions p 90', 'Goal creating actions p 90']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18243 entries, 0 to 18242
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Total Shots                 18243 non-null  int64  
 1   % Shots on target           18243 non-null  float64
 2   Shots p 90                  18243 non-null  float64
 3   Goals per shot              18243 non-null  float64
 4   Goals per shot on target    18243 non-null  float64
 5   % Aerial Duels won          18243 non-null  float64
 6   Shot creating actions p 90  18243 non-null  float64
 7   Goal creating actions p 90  18243 non-null  float64
dtypes: float64(7), int64(1)
memory usage: 1.1 MB


No preprocessing needed

### 3. Export the Preprocessed Data

In [ ]:
df_preprocessed.to_csv('./datasets/preprocessed_data.csv', index=False)